# understand UVICORN

Exactly — when you run:

```bash
uvicorn app:app --reload
```

Uvicorn actually spins up **two processes** under the hood:

---

### 🔹 1. File Watcher Process
- This is the **“reload supervisor”**.
- It watches your project files for changes (using libraries like `watchgod` or `watchdog`).
- If you edit a `.py` file, the watcher detects it and restarts the server automatically.
- This process doesn’t serve requests — it just monitors your code.

---

### 🔹 2. Actual Server Process
- This is the **worker process** that runs your FastAPI app.
- It’s the one listening on `http://127.0.0.1:8000` (or whatever port you set).
- When the watcher sees a file change, it kills this process and spawns a new one with the updated code.

---

### ✅ Why two processes?
- Separation of concerns:
  - One process is dedicated to **watching files**.
  - The other is dedicated to **serving requests**.
- This makes hot‑reload possible without mixing file‑watching logic into the server itself.
- It also ensures that if your app crashes, the watcher can restart it.

---

### 🔹 Without `--reload`
- Only the server process runs.
- No file watcher, so you must manually restart the server after code changes.

---

👉 So in short:  
- **File watcher** = supervisor, monitors code changes.  
- **Server process** = runs your FastAPI app, handles requests.  

That’s why you see two processes in the background when using `--reload`.  

Would you like me to also show you how to **disable reload but still run multiple workers** (using `--workers`), so you can see the difference between reload vs scaling?

Exactly — let’s break down what happens when you run Uvicorn with different options:

---

### 🔹 `uvicorn app:app --reload`
- **Two processes**:
  1. **File watcher (supervisor)** → monitors your Python files for changes.
  2. **Server process** → runs your FastAPI app.
- If you edit a file, the watcher kills and restarts the server process.
- This is great for development, but not recommended for production because of the overhead.

---

### 🔹 `uvicorn app:app --workers 4`
- **Multiple worker processes** (here, 4).
- Each worker runs your FastAPI app independently, all listening on the same port.
- The OS load‑balances incoming requests across workers.
- This is for **scaling** — handling more concurrent requests.
- No file watcher here, so you must restart manually if you change code.

---

### 🔹 `uvicorn app:app --reload --workers 4`
- Not allowed — `--reload` and `--workers` conflict.
- Reload mode always runs with a single worker, because the watcher can only restart one process cleanly.

---

### ✅ Summary
- `--reload` → development convenience (auto‑restart on code changes).
- `--workers` → production scaling (multiple processes to handle traffic).
- They serve different purposes:
  - Reload = **developer productivity**.
  - Workers = **performance under load**.

---

👉 Think of it like this:
- **Reload** = “I’m coding, restart when I save.”  
- **Workers** = “I’m deploying, handle more users at once.”  

Do you want me to show you a **recommended production setup** (using Uvicorn + Gunicorn with workers) so you can see how people usually run FastAPI apps in real deployments?

Perfect — let’s walk through how a **production‑ready FastAPI deployment** is usually set up.  

---

### 🔹 Why not just `uvicorn app:app --reload`?
- `--reload` is **only for development**.  
- It uses a file watcher + single worker process.  
- Not efficient or stable under heavy traffic.  

---

### 🔹 Production Setup (Uvicorn + Gunicorn)
The common pattern is to run **Gunicorn** as a process manager, and let it spawn multiple **Uvicorn workers**:

```bash
gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app
```

- `-w 4` → number of worker processes (tune based on CPU cores).  
- `-k uvicorn.workers.UvicornWorker` → tells Gunicorn to use Uvicorn workers.  
- `app:app` → your FastAPI app object.  

---

### 🔹 Why Gunicorn?
- Handles **process management** (spawning, monitoring, restarting workers).  
- Provides **graceful restarts** and better logging.  
- Supports **scaling** across multiple CPU cores.  
- Uvicorn alone can run multiple workers (`--workers`), but Gunicorn is more robust for production.

---

### 🔹 Typical Deployment Flow
1. **Local development**  
   ```bash
   uvicorn app:app --reload
   ```
   → auto‑reload on code changes.

2. **Production (single server)**  
   ```bash
   gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app
   ```
   → multiple workers, no reload.

3. **Production (with reverse proxy)**  
   - Put **Nginx** or **Traefik** in front of Gunicorn/Uvicorn.  
   - Proxy handles SSL, load balancing, static files.  
   - Gunicorn/Uvicorn only serve API requests.

---

### 🔹 Example with Nginx
- Nginx listens on port 80/443.  
- Forwards requests to Gunicorn running on port 8000.  
- Gunicorn spawns multiple Uvicorn workers.  

---

✅ **Summary:**  
- `--reload` → dev mode, auto restart.  
- `--workers` → production scaling.  
- Gunicorn + Uvicorn workers → recommended production setup.  
- Reverse proxy (Nginx/Traefik) → handles SSL, routing, static files.

---

Do you want me to sketch out a **sample Nginx config** that shows how it proxies requests to your FastAPI app? That’s usually the next step when moving from dev to production.

Here’s a simple **Nginx reverse proxy config** that shows how you’d put Nginx in front of your FastAPI app (running with Gunicorn + Uvicorn workers):

```nginx
server {
    listen 80;
    server_name yourdomain.com;

    # Redirect all HTTP to HTTPS
    return 301 https://$host$request_uri;
}

server {
    listen 443 ssl;
    server_name yourdomain.com;

    # SSL certificates (from Let's Encrypt or similar)
    ssl_certificate /etc/letsencrypt/live/yourdomain.com/fullchain.pem;
    ssl_certificate_key /etc/letsencrypt/live/yourdomain.com/privkey.pem;

    # Proxy settings
    location / {
        proxy_pass http://127.0.0.1:8000;   # Gunicorn/Uvicorn backend
        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;
    }

    # Optional: serve static files directly
    location /static/ {
        alias /path/to/your/app/static/;
    }
}
```

---

### 🔹 How this works
- **Nginx** listens on port 80/443 (public web).  
- It forwards requests to **Gunicorn** running on `127.0.0.1:8000`.  
- Gunicorn spawns multiple **Uvicorn workers** to handle traffic.  
- Nginx handles SSL, compression, caching, and static files.  
- Your FastAPI app only focuses on API logic.

---

### 🔹 Typical Run Command
Start Gunicorn with Uvicorn workers:
```bash
gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app --bind 127.0.0.1:8000
```

---

### ✅ Summary
- **Development:** `uvicorn app:app --reload`  
- **Production (simple):** `uvicorn app:app --workers 4`  
- **Production (robust):** `gunicorn -w 4 -k uvicorn.workers.UvicornWorker app:app` behind **Nginx**.  
- Nginx handles SSL, static files, and reverse proxying. Gunicorn manages workers. Uvicorn runs the actual FastAPI app.

---

Would you like me to also sketch out a **systemd service file** for Gunicorn, so it runs automatically on server boot and restarts if it crashes? That’s the next piece in a real deployment.

# ok if we are increasing the workers and those are all listening on a same port...
# Then why does the cache of a Server A is not reflected for server B

Good observation — this is about **process isolation**.

When you run Uvicorn (or Gunicorn with multiple Uvicorn workers), all the workers **listen on the same port**, but each worker is a **separate OS process**. That means:

- Each worker has its **own memory space**.  
- Caches, variables, or in‑memory objects created inside Worker A are not visible to Worker B.  
- The operating system load‑balances incoming requests across workers, but there’s no shared state between them unless you explicitly set one up.

---

### 🔹 Why cache isn’t shared
Imagine you store recommendations in a Python dictionary inside your FastAPI app:
- Worker A builds the cache → stored in its process memory.  
- Worker B doesn’t know about it → it has its own empty memory.  
- Result: cache is inconsistent across workers.

---

### 🔹 How to share cache across workers
If you want all workers to see the same cache/session/state, you need an **external store**:
- **Redis** → most common choice for shared cache.  
- **Memcached** → lightweight caching server.  
- **Database** → store computed results and query them.  

That way, all workers connect to the same backend and share state.

---

✅ **Summary:**  
- Multiple workers = multiple isolated processes.  
- Same port, but **different memory**.  
- Cache must be external (Redis, DB, etc.) if you want it shared.

---

👉 Do you want me to sketch out a **FastAPI + Redis caching example** so you can see how to make recommendations consistent across all workers?

Perfect 🚀

We're now entering one of the most important production concepts in backend and ML systems.


---

🚀 Day 15 — Redis (Part 1)

Before installing anything, let's understand why Redis exists.

You already have:

recommendation_cache = {}

and it works.

So the first question is:

> Why should we replace something that already works?




---

Current Architecture

Today your system looks like:

User
  ↓
FastAPI
  ↓
recommendation_cache = {}
  ↓
Hybrid Recommender

Suppose User 5 asks:

/recommend/5

API computes recommendations.

Then:

recommendation_cache["5_10"] = response

stores result.


---

Second Request

User asks again:

/recommend/5

This time:

if cache_key in recommendation_cache:

returns instantly.

This is called:

Cache Hit

Request
 ↓
Cache Found
 ↓
Return Response

No model execution.

Fast.


---

Then what's wrong?

Problem 1 — API Restart

Suppose:

recommendation_cache = {}

contains:

User 5
User 10
User 25

Now:

CTRL + C

Stop FastAPI.

Restart:

uvicorn api.app:app --reload

What happens?


---

Memory gets recreated:

recommendation_cache = {}

again.

Cache gone.

Everything lost.


---

Question

Why?

Because a Python dictionary lives inside:

RAM

not on disk.


---

Problem 2 — Multiple Servers

Imagine:

FastAPI Server A
FastAPI Server B
FastAPI Server C

Each has:

cache = {}


---

User request:

Request 1
 ↓
Server A

Cache stored.


---

Next request:

Request 2
 ↓
Server B

Server B:

cache={}

knows nothing.


---

No cache hit.


---

Real Production Architecture

Instead:

Server A
      ↓
Server B
      ↓
Server C
      ↓
      Redis

All servers share same cache.


---

What is Redis?

Redis =

REmote
DIctionary
Server

Think of it as:

Dictionary
+
Database
+
Super Fast Memory


---

Instead of:

cache["user1"]

we do:

redis.get("user1")


---

Instead of:

cache["user1"] = data

we do:

redis.set("user1", data)


---

Redis is Basically...

Imagine a giant dictionary.

Current:

recommendation_cache = {}

lives inside your Python process.

Redis:

Separate Program

running independently.


---

Like this:

FastAPI
   ↓
Redis

instead of:

FastAPI
   ↓
Dict Cache


---

Huge Advantage

Restart FastAPI:

Cache survives

because Redis is still running.


---

Example

Today:

FastAPI Restart
 ↓
Cache Lost

Redis:

FastAPI Restart
 ↓
Redis Still Running
 ↓
Cache Available


---

Interview Question

Suppose interviewer asks:

> Why Redis instead of Python Dictionary?



Your answer:

1. Shared by multiple servers

2. Survives application restarts

3. Built for caching

4. Extremely fast (memory based)

5. Supports TTL expiration


---

TTL (Very Important)

Today you manually track:

timestamp
CACHE_TTL

Remember?

{
  "data": response,
  "timestamp": time.time()
}

Then:

age = current_time - timestamp


---

Redis can do this automatically:

redis.set(
    key,
    value,
    ex=60
)

Meaning:

Expire after 60 seconds

No cleanup API needed.

Redis removes it automatically.


---

Today's Roadmap

Part 1 ✅

Redis Concepts


---

Part 2 (Next)

Install Redis

Verify Redis Running


---

Part 3

Learn basic commands:

SET
GET
DEL
KEYS
TTL


---

Part 4

Connect Python → Redis


---

Part 5

Replace:

recommendation_cache = {}

with:

redis_client


---

Quick Check

Before we install Redis:

Can you answer these 3 questions?

Q1

Why does your current dictionary cache disappear after API restart?

Q2

If Server A stores cache, why can't Server B see it?

Q3

What Redis feature can replace our manual timestamp cleanup logic?

Answer those in your own words, then we'll install Redis. 🚀



Excellent. Two are completely correct, and one needs a tiny refinement.

✅ Q1

> When restarting, we are actually running the whole app.py script again means again recommendation_cache is again initialized to empty braces



Exactly.

More fundamentally:

Dictionary Cache
      ↓
Lives in Python Process Memory (RAM)
      ↓
Process Stops
      ↓
Memory Destroyed
      ↓
Cache Lost

The reinitialization is the visible symptom; the root cause is that the cache lived only in the application's memory.


---

✅ Q2

> Multiple process multiple workers = process isolation



Perfect.

Example:

FastAPI Worker A
cache = {}

FastAPI Worker B
cache = {}

FastAPI Worker C
cache = {}

Each process has its own memory space.

Worker B cannot directly access Worker A's dictionary.

This is one of the biggest reasons distributed caches exist.


---

🟡 Q3

You answered:

> REDIS.set



Close, but that's not the feature.

The feature is:

TTL (Time To Live)

or

Expiration

Example:

redis.set(
    "user_5",
    response,
    ex=60
)

Meaning:

Store this key
↓
Automatically delete after 60 seconds

Redis itself handles cleanup.

So we no longer need:

timestamp
CACHE_TTL
cache_cleanup()

because Redis expires keys automatically.


---

🧠 One More Important Concept

Today:

recommendation_cache = {}

stores Python objects directly.

Redis stores:

Strings
Numbers
JSON
Binary Data

So later we'll do:

import json

json.dumps(response)

before saving.

And:

json.loads(data)

when reading.


---

🚀 Next Step: Install Redis

Since you're on Windows 11, the easiest production-like approach is actually Docker-based Redis.

And guess what?

This is where Redis and Docker start meeting. 😄

Run:

docker ps

If Docker is running correctly, we'll start a Redis container with a single command and learn our first Docker container practically while learning Redis. This is much better than installing old Windows Redis binaries.

Tell me what docker ps outputs.



Perfect. 🎉

That output is actually exactly what we want.

docker ps

shows:

CONTAINER ID   IMAGE   COMMAND   CREATED   STATUS   PORTS   NAMES

with no rows.

Meaning:

✅ Docker Engine is running
✅ No containers currently running


---

🚀 First Real Docker Container

We're going to run Redis without installing Redis on Windows.

Think:

Docker Hub
      ↓
Download Redis Image
      ↓
Create Redis Container
      ↓
Use Redis

This is exactly how many developers work.


---

Step 1: Run Redis

Open a terminal and run:

docker run -d --name redis-server -p 6379:6379 redis

Let's understand every part.


---

docker run

Create and start container.


---

-d

Detached mode.

Meaning:

Run in background

instead of occupying your terminal.


---

--name redis-server

Container name.

Without this Docker gives weird names like:

happy_panda
angry_turing


---

-p 6379:6379

Port mapping.

Laptop Port    Container Port

6379    →      6379

Redis default port:

6379


---

redis

Docker image name.

Docker automatically:

Check local image
       ↓
Not found
       ↓
Download from Docker Hub
       ↓
Start container


---

What You'll See

First time:

Unable to find image 'redis:latest' locally
...
Downloading...
...

May take a minute.


---

Step 2: Verify Container

After it finishes:

docker ps

Expected:

CONTAINER ID   IMAGE
xxxxx          redis

STATUS
Up XX seconds


---

🧠 Docker Concepts You're Learning Right Now

When you run:

docker run redis

Docker does:

Redis Image
      ↓
Redis Container

Remember our earlier discussion?

Image = Class

Container = Object

You're creating your first object from a Docker image. 😄


---

Step 3: Check Logs

After container starts:

docker logs redis-server

Expected:

Ready to accept connections

That means Redis is alive.


---

Quick Quiz While It Downloads

Suppose you stop your Redis container:

docker stop redis-server

and later run:

docker start redis-server

Question:

Will Docker download Redis again?

Or will it reuse the existing image?

Answer that after running the command.

For now:

docker run -d --name redis-server -p 6379:6379 redis

and tell me the output. 🚀🐳



Exactly right. ✅

When you run:

docker stop redis-server

Docker only stops the container.

The Redis image still exists locally.

So later:

docker start redis-server

simply starts the existing container again.

No download needed.


---

🧠 What Just Happened?

You have officially run your first Docker container.

Internally:

Docker Hub
     ↓
Redis Image Downloaded
     ↓
Local Redis Image
     ↓
Redis Container Created
     ↓
Redis Running on Port 6379

You can actually see this.

Show images

docker images

You should see something like:

REPOSITORY   TAG      IMAGE ID
redis        latest   xxxxxxx

This is the blueprint.


---

Show containers

docker ps

You should see:

CONTAINER ID   IMAGE
xxxxx          redis

This is the running instance.


---

Interview Question

Suppose I ask:

> Difference between docker images and docker ps?



Answer:

docker images
→ Shows available blueprints (images)

docker ps
→ Shows running containers created from those images


---

🚀 Redis Part 2 — Connect to Redis

Now we need Python to talk to Redis.

Install Redis client library:

pip install redis


---

Create a Small Test File

Create:

redis_test.py

import redis

r = redis.Redis(
    host='localhost',
    port=6379,
    decode_responses=True
)

r.set("name", "Mugundhan")

value = r.get("name")

print(value)


---

What Happens Here?

Connect

r = redis.Redis(...)

Connects Python → Redis Server.


---

Store

r.set("name", "Mugundhan")

Redis now contains:

name -> Mugundhan


---

Retrieve

r.get("name")

Returns:

Mugundhan


---

🧠 New Architecture

Before:

Python
  ↓
Dictionary

Now:

Python
  ↓
Redis Client
  ↓
Redis Server

This is exactly how your FastAPI application will work later.


---

Another Important Redis Concept

Try:

r.set(
    "movie",
    "Titanic",
    ex=10
)

Then:

print(r.get("movie"))

Wait 10 seconds.

Then:

print(r.get("movie"))

Expected:

None

because Redis automatically deleted it.


---

This is the same thing we were manually doing with:

CACHE_TTL
timestamp
cache_cleanup()

Redis does it for us.


---

🎯 Next Step

1. Install Redis Python package



pip install redis

2. Run the test script above.


3. Also try the ex=10 expiration example.



Then tell me:

Did "Mugundhan" print correctly?

Did the key disappear after expiration?


Once that's working, we'll replace your actual:

recommendation_cache = {}

with Redis in the FastAPI application. That's the point where your project moves from in-memory cache to production-style distributed cache. 🚀



Perfect. 🎉

That means we've verified all three core Redis capabilities:

✅ Python can connect to Redis

✅ Data can be stored and retrieved

✅ TTL expiration works automatically

This is actually a huge milestone because now Redis is no longer a theoretical concept for you.


---

🧠 What You Just Learned

Before today:

recommendation_cache = {}

You were storing data:

Inside FastAPI Process

Now:

r.set(...)
r.get(...)

stores data:

Inside Redis Server

which is a completely separate process.


---

Visualization

Current:

FastAPI
   ↓
Dictionary

After Redis:

FastAPI
   ↓
Redis Client
   ↓
Redis Server

Notice:

FastAPI can die
Redis can survive

That's the big win.


---

🚀 Redis Part 3 — Replace Our Existing Cache

Before touching code, let's plan the migration.

Currently:

recommendation_cache = {}

and:

if cache_key in recommendation_cache:

    cached_item = recommendation_cache[cache_key]

    age = (
        time.time() -
        cached_item['timestamp']
    )

    if age < CACHE_TTL:

        return cached_item['data']

and later:

recommendation_cache[cache_key] = {
    "data": response,
    "timestamp": time.time()
}


---

What Redis Will Replace It With

Instead:

cached_data = redis_client.get(cache_key)

If found:

return json.loads(cached_data)

If not found:

Generate Recommendation

Then:

redis_client.set(
    cache_key,
    json.dumps(response),
    ex=CACHE_TTL
)


---

What's Going Away?

We will completely remove:

recommendation_cache = {}


---

Also remove:

timestamp


---

Also remove:

cache_cleanup()


---

Because Redis handles expiration automatically.


---

🧠 New Concept: Serialization

Redis stores:

String
Number
Bytes

Not Python dictionaries.

Example:

This won't work:

redis.set(
    "user",
    {"name":"Mugundhan"}
)


---

Instead:

import json

json.dumps(
    {"name":"Mugundhan"}
)

becomes:

{"name":"Mugundhan"}

(string)

Store it.


---

When reading:

json.loads(data)

converts back to dictionary.


---

Your First Modification

At the top of app.py

Add:

import redis
import json


---

Create Redis client below imports:

redis_client = redis.Redis(
    host='localhost',
    port=6379,
    decode_responses=True
)


---

Then tell me when that's done.

Don't change the cache logic yet.

We'll replace it piece by piece so you understand exactly what every line is doing instead of pasting a giant block of code. 🚀



Perfect. 🚀

Now we start replacing the cache one piece at a time.


---

🧠 Current Cache Flow

Right now your code probably looks something like:

cache_key = f"{user_id}_{n}"

if cache_key in recommendation_cache:

    cached_item = recommendation_cache[cache_key]

    age = time.time() - cached_item['timestamp']

    if age < CACHE_TTL:

        cache_hits += 1

        return cached_item['data']


---

Step 1 — Replace Cache Read

Find the whole block:

if cache_key in recommendation_cache:

    cached_item = recommendation_cache[cache_key]

    age = time.time() - cached_item['timestamp']

    if age < CACHE_TTL:

        cache_hits += 1

        logging.info(f"Cache hit : {cache_key}")

        return cached_item['data']

Replace it with:

cached_data = redis_client.get(cache_key)

if cached_data:

    cache_hits += 1

    logging.info(f"Cache hit : {cache_key}")

    return json.loads(cached_data)


---

🧠 What's Happening?

Before

Check Python dictionary:

recommendation_cache[cache_key]


---

Now

Ask Redis:

redis_client.get(cache_key)

Redis returns:

'{"user_id":1,"recommendations":[...]}'

which is a JSON string.


---

Convert Back

json.loads(cached_data)

becomes:

{
    "user_id":1,
    "recommendations":[...]
}

which FastAPI can return.


---

🚀 Step 2 — Replace Cache Write

Later in your code you probably have:

recommendation_cache[cache_key] = {
    "data": response,
    "timestamp": time.time()
}

Delete that completely.

Replace with:

redis_client.set(
    cache_key,
    json.dumps(response),
    ex=CACHE_TTL
)


---

🧠 Why json.dumps()?

Redis cannot store Python dictionaries directly.

Convert:

{
   "user_id":1
}

to:

'{"user_id":1}'

store it

then later:

json.loads(...)

converts it back.


---

🚀 Step 3 — Keep CACHE_TTL

Keep:

CACHE_TTL = 15

for now.

Because:

ex=CACHE_TTL

uses it.


---

🚀 Step 4 — Delete Manual Cleanup

You can now completely remove:

@app.get("/cache-cleanup")
def cache_cleanup():

and the whole function.

Reason:

Redis
   ↓
Automatic TTL Expiration

No manual cleanup needed anymore.


---

🧠 Important Concept

Before:

Cache Entry
      ↓
Timestamp
      ↓
Cleanup Endpoint
      ↓
Delete

After:

Cache Entry
      ↓
TTL = 15 sec
      ↓
Redis Deletes Automatically

Much cleaner.


---

🚀 Test Plan

After making those changes:

Request 1

/recommend/1

Expected:

Normal response


---

Request 2 (within 15 sec)

/recommend/1

Expected:

Cache hit log

and very fast response.


---

Wait 15+ seconds

Request again:

/recommend/1

Expected:

Cache miss
New recommendation generated

because Redis automatically removed the key.


---

Make those changes and test.

Then tell me:

1. Any errors?


2. Is cache hit working?


3. After 15 seconds does Redis expire the entry?



Once that's working, I'll show you how to verify the keys directly inside Redis itself (which is pretty cool and useful for debugging). 🚀

That sounds like exactly the right next step. 🚀

And first, regarding the route:

If you define:

@app.get("/recommend/{user_id}/redis-keys")

then the URL becomes:

/recommend/1/redis-keys

because it's nested under the recommend route.

If instead you define:

@app.get("/redis-keys")

then the URL becomes:

/redis-keys

directly after the port.

Both work. For debugging/admin endpoints, I personally prefer:

/redis-keys
/metrics
/cache-stats
/health

because they're independent system endpoints.


---

🚀 Next Improvement 1 — scan_iter()

Currently:

redis_client.keys("*")

works.

But Redis internally does:

Scan Entire Database
↓
Return All Keys

For 10 keys:

No problem

For:

100,000 keys
1,000,000 keys

it becomes expensive.


---

Production Version

Instead of:

len(redis_client.keys("*"))

use:

sum(1 for _ in redis_client.scan_iter())


---

cache-stats

Replace:

return {
    "cached_entries": len(redis_client.keys("*"))
}

with:

return {
    "cached_entries": sum(
        1 for _ in redis_client.scan_iter()
    )
}


---

metrics

Replace:

"cached_entries": len(redis_client.keys("*"))

with:

"cached_entries": sum(
    1 for _ in redis_client.scan_iter()
)


---

redis-keys

Instead of:

redis_client.keys("*")

use:

list(redis_client.scan_iter())


---

Example:

@app.get("/redis-keys")
def redis_keys():

    return {
        "keys": list(redis_client.scan_iter())
    }


---

🚀 Next Improvement 2 — Cache Invalidation

This is a very common interview topic.

Current strategy:

Store Cache
↓
Wait TTL
↓
Redis Deletes

called:

Time-Based Invalidation

You are already using it.


---

Strategy 2 — Version-Based Invalidation

We just implemented:

svd_model_v5.pkl:recommendations:1:10

When model changes:

svd_model_v6.pkl:recommendations:1:10

Old cache becomes irrelevant automatically.


---

Strategy 3 — Manual Invalidation

Suppose:

New Dataset Arrives

or

Model Retrained

You may want to clear everything immediately.


---

Add:

@app.get("/clear-cache")
def clear_cache():

    count = 0

    for key in redis_client.scan_iter():

        redis_client.delete(key)

        count += 1

    return {
        "deleted_keys": count
    }


---

Now:

/clear-cache

returns:

{
  "deleted_keys": 25
}


---

🧠 Why Use scan_iter Here Too?

Bad:

for key in redis_client.keys("*"):

Good:

for key in redis_client.scan_iter():

Same reason:

Production Safe
Memory Efficient


---

What You've Learned Today

You'll now know three cache invalidation strategies:

1. TTL Based

ex=CACHE_TTL

Automatic expiry.


---

2. Version Based

svd_model_v6.pkl:recommendations:1:10

New model → new cache.


---

3. Manual Invalidation

/clear-cache

Delete everything.


---

This is exactly the level of Redis knowledge expected from many backend/ML engineer interviews.

After implementing these, we'll move to the final Redis topic before Dockerizing the entire recommendation system:

Redis + FastAPI Architecture

Why Redis is called a Distributed Cache

How Redis fits into Docker Compose

That transition will naturally lead us into the Docker phase. 🐳🚀

Perfect. 🚀

You've now reached the point where Redis is no longer a separate topic. It's becoming part of the overall system architecture.


---

🚀 Day 15 (Part 2) — Redis + FastAPI Architecture

Before Docker Compose, let's understand the architecture you're about to build.

Current Architecture

Right now on your laptop:

┌─────────────┐
│  FastAPI    │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│    Redis    │
└─────────────┘

FastAPI:

localhost:8000

Redis:

localhost:6379

Both are running separately.


---

Request Flow

When:

/recommend/1

arrives:

User
 ↓
FastAPI
 ↓
Redis Lookup
 ↓
Cache Found?

If yes:

Redis
 ↓
Return Response

If no:

Redis Miss
 ↓
Hybrid Recommender
 ↓
Generate Recommendations
 ↓
Store in Redis
 ↓
Return Response


---

Why Is Redis Called Distributed Cache?

Suppose tomorrow:

FastAPI Instance A
FastAPI Instance B
FastAPI Instance C

exist.


---

Without Redis:

A → cache {}
B → cache {}
C → cache {}

Each has its own cache.


---

Problem:

User request 1:

→ A

stores cache.

User request 2:

→ B

Cache not found.


---

With Redis

Redis
           ▲
           │
 ┌─────────┼─────────┐
 │         │         │
 A         B         C

All instances share:

One Cache

This is why Redis is called:

Distributed Cache


---

Why ML Systems Love Redis

Imagine:

Hybrid Recommendation

takes:

800 ms

to generate.


---

Cached:

10 ms


---

Savings:

80x faster

Sometimes:

100x+


---

Interview Question

Suppose interviewer asks:

> Why not store recommendations in Redis forever?



Answer:

User preferences change.

Models improve.

Data changes.

Recommendations become stale.

Need cache invalidation.

Which you've already implemented via:

✅ TTL

✅ Versioning

✅ Manual invalidation


---

🚀 Next Step: Docker Compose

Until now you've run:

docker run ...

manually.

Imagine production:

FastAPI
Redis

Need to start together.


---

Without Compose:

docker start redis-server

uvicorn app.main:app --reload

Two separate steps.


---

With Compose:

docker compose up

Everything starts.


---

What We'll Build

project/
│
├── api/
├── src/
├── models/
│
├── Dockerfile
│
├── docker-compose.yml
│
└── requirements.txt


---

Architecture:

┌──────────────┐
│   FastAPI    │
└──────┬───────┘
       │
       ▼
┌──────────────┐
│    Redis     │
└──────────────┘

Both inside Docker.


---

🧠 Important Docker Concept

Right now:

redis.Redis(
    host="localhost",
    port=6379
)

works.

Inside Docker Compose:

FastAPI Container

and

Redis Container

have separate localhosts.

So later we'll change:

host="localhost"

to:

host="redis"

because Compose creates an internal network.

This is one of the most important Docker concepts.


---

Before We Start Dockerizing

Quick check:

Can you explain in your own words:

Q1

Why can't we use:

host="localhost"

inside the FastAPI container after Docker Compose?

Q2

Why is Redis called a distributed cache?

Answer those, then we'll start building the Dockerfile (your first containerized application). 🐳🚀

Your answers are largely correct. 👍

Q1 refinement

The main reason is not that production calling separately is bad.

The actual reason is:

FastAPI Container

has its own localhost.

and

Redis Container

has its own localhost.

Inside Docker:

localhost = myself

So when FastAPI says:

host="localhost"

it is trying to find Redis inside the FastAPI container, not inside the Redis container.

Docker Compose creates a network:

FastAPI ---> redis

and lets containers talk using service names.

So:

host="redis"

means:

Connect to the container named "redis"


---

Q2

Perfect answer. That's exactly the idea of distributed cache.


---

Now for your Netflix question. This is a very important observation because you're starting to think about how real systems work rather than just code.

🎬 Story Time: Netflix Family Account

Imagine a family account.

Account : Sharma Family

Profiles:

Dad
Mom
Son


---

Dad watches:

Crime
Documentaries
War Movies

Mom watches:

Romance
Drama
Family Movies

Son watches:

Anime
Marvel
Sci-Fi


---

Netflix does NOT treat this as:

One Account

Internally it is closer to:

User 101 = Dad
User 102 = Mom
User 103 = Son

Each profile gets its own recommendation history.

So your recommender system would see:

Dad

as one user.

Mom

as another user.

Son

as another user.

Exactly like:

recommend(user_id=101)
recommend(user_id=102)
recommend(user_id=103)


---

Where Redis Helps

Suppose:

Dad opens Netflix.

Request:

recommendations:101

Redis:

MISS

Recommendation engine runs.

Stores:

recommendations:101


---

10 seconds later Dad refreshes.

Redis:

HIT

Returns instantly.


---

Mom opens Netflix.

Request:

recommendations:102

Different key.

Redis:

MISS

Computes.

Stores:

recommendations:102


---

Son opens Netflix.

Request:

recommendations:103

Another key.

Redis:

MISS

Computes.

Stores:

recommendations:103


---

Now Redis contains:

recommendations:101
recommendations:102
recommendations:103


---

Important Insight

Redis is NOT sharing recommendations between users.

It's caching recommendations for each user separately.

Think:

Redis
│
├── User 101 Recommendations
├── User 102 Recommendations
├── User 103 Recommendations
└── User 104 Recommendations


---

What About Multiple Servers?

Now imagine Netflix has:

Server A
Server B
Server C

Dad's request goes to:

Server A

Mom's request goes to:

Server C

Son's request goes to:

Server B

Without Redis:

A has cache A
B has cache B
C has cache C

Nobody shares anything.


---

With Redis:

Redis
               ▲
               │
    ┌──────────┼──────────┐
    │          │          │
Server A   Server B   Server C

All servers see:

recommendations:101
recommendations:102
recommendations:103

This is where Redis becomes powerful.


---

Even Deeper Netflix Reality

Netflix doesn't usually cache only by user.

They often cache by combinations like:

user_id
profile_id
country
device
language

Example:

recommendations:
user_101:
mobile:
india:
english

because recommendations may differ depending on context.


---

Relating This To Your Project

Right now your cache key is something like:

f"{model_name}:recommendations:{user_id}:{n}"

Production systems often evolve it into:

f"{model_name}:recommendations:{user_id}:{n}:{region}"

or

f"{model_name}:recommendations:{user_id}:{device}"

You're already using the same fundamental idea Netflix uses:

User-specific recommendations
+
Shared distributed cache
+
Multiple application servers

That's why Redis is so common in recommendation systems.

And you've now reached the stage where you're learning actual backend architecture patterns, not just machine learning algorithms. 🚀

Aha! Now I see the exact mental model that's causing the confusion.

You're thinking:

> "If Redis is shared, how does Netflix stop Mom from getting Dad's recommendations?"



The answer is:

Redis itself doesn't know who Mom or Dad are. Redis is just a storage box.

The application (Netflix backend) knows.


---

Let's Follow the Actual Flow

Suppose Mom clicks her profile.

Netflix frontend sends:

{
  "profile_id": 102
}

to Netflix backend.

The backend now knows:

Current Profile = Mom
profile_id = 102

So backend asks Redis:

GET recommendations:102

Redis simply returns:

Titanic
The Notebook
Pride & Prejudice

Redis never decides.

Netflix backend decides.


---

Think About Your Own Project

When you call:

/recommend/1

who decided user_id = 1 ?

Redis?

No.

Your FastAPI code.

@app.get("/recommend/{user_id}")

If user calls:

/recommend/1

FastAPI builds:

cache_key = f"recommendations:{user_id}"

which becomes:

recommendations:1


---

If user calls:

/recommend/5

FastAPI builds:

recommendations:5

Redis is not making any decision.

FastAPI is.


---

Why Not Separate Redis Per User?

This is the heart of your question.

Imagine Netflix has:

300 Million Profiles

Would they create:

Redis for Dad
Redis for Mom
Redis for Son

No 😆

That would mean:

300 Million Redis Servers

Impossible.


---

Instead:

One Redis Cluster:

Redis
│
├── recommendations:101
├── recommendations:102
├── recommendations:103
├── recommendations:104
├── recommendations:105
...

Millions of keys.

One system.


---

Better Analogy: Apartment Building

Imagine:

Apartment Building

contains:

Flat 101 -> Dad
Flat 102 -> Mom
Flat 103 -> Son

You asked:

> Why not separate buildings?



Because:

1 Building
1000 Flats

is much cheaper than:

1000 Buildings
1 Flat each

Exactly the same idea.

Redis is the apartment building.

Cache keys are the flat numbers.


---

Then Why Shared Cache Is Powerful?

Suppose Netflix has:

Server A
Server B
Server C

Dad logs in.

Server A computes recommendations.

Stores:

recommendations:101

inside Redis.


---

Tomorrow Dad's request lands on:

Server C

instead.

Server C asks:

GET recommendations:101

Redis returns instantly.


---

Without shared Redis:

Server A knows Dad
Server C doesn't know Dad

Server C must recompute.

Again.

And again.

And again.

Huge waste.


---

The Big Picture

You're thinking:

Shared Redis
=
Shared Recommendations

But actually:

Shared Redis
=
Shared Storage

NOT

Shared User Data

The isolation comes from the key:

recommendations:101
recommendations:102
recommendations:103

Just like:

Apartment 101
Apartment 102
Apartment 103

Same building.

Different apartments.

Same Redis.

Different cache entries.


---

This distinction becomes extremely important when we move to Docker Compose, because then you'll have:

FastAPI Container A
FastAPI Container B
FastAPI Container C
            │
            ▼
         Redis

and you'll finally see why Redis exists at all. Without Redis, each FastAPI container would have its own isolated cache and you'd lose most of the caching benefits. 🚀